# Surgeon Validation Stratified Sample 30

Thin workflow notebook. All sampling and clip extraction logic lives in `src/cvs_act`.

In [1]:
from pathlib import Path

from cvs_act.clip_extraction import assert_audit_v11_consistency, select_one_clip_per_video, write_clips
from cvs_act.surgeon_sampling import (
    SEED,
    SURGEON_ANNOTATION_ROOT,
    add_collapsed_country,
    compute_poststrat_weights,
    exclude_audit_v11,
    load_audit_v11_ids,
    load_eligible_pool,
    per_cell_counts,
    stratified_sample,
    summarize_marginals,
)


In [2]:
eligible_168 = load_eligible_pool(verify_hf=False)
audit_ids = load_audit_v11_ids()
pool_138 = exclude_audit_v11(eligible_168, audit_ids)

sample_30 = stratified_sample(pool_138, n=30, seed=SEED)
weights = compute_poststrat_weights(sample_30, pool_138)
sample_30 = sample_30.assign(weight=weights)
design_effect = weights.attrs['design_effect']

assert_audit_v11_consistency(audit_ids[:5])
SURGEON_ANNOTATION_ROOT.mkdir(parents=True, exist_ok=True)
sample_path = SURGEON_ANNOTATION_ROOT / 'sample_30.csv'
sample_cols = [
    'video_name', 'c1_bit', 'c2_bit', 'c3_bit', 'stratum',
    'country', 'country_collapsed', 'ioc', 'icg', 'robotic', 'device', 'weight',
]
sample_30[sample_cols].to_csv(sample_path, index=False)
clip_manifest = write_clips(sample_30)
clip_manifest_path = SURGEON_ANNOTATION_ROOT / 'clip_manifest.csv'
clip_manifest.to_csv(clip_manifest_path, index=False)
selected_video_clips = select_one_clip_per_video(clip_manifest, seed=SEED, granularity='coarse')
selected_clip_path = SURGEON_ANNOTATION_ROOT / 'selected_video_clips.csv'
selected_video_clips.to_csv(selected_clip_path, index=False)
print('sample:', sample_path)
print('clip manifest:', clip_manifest_path)
print('selected clips:', selected_clip_path)
print('clip JSON root:', SURGEON_ANNOTATION_ROOT / 'clips')
print('design effect:', round(design_effect, 3), 'HIGH' if design_effect > 2 else 'ok')


sample: /mnt/md0/weiqiuy/surgent/data/processed/CVS_Challenge_SAGES_v1/cvs_act_surgeon_annotations/sample_30.csv
clip manifest: /mnt/md0/weiqiuy/surgent/data/processed/CVS_Challenge_SAGES_v1/cvs_act_surgeon_annotations/clip_manifest.csv
selected clips: /mnt/md0/weiqiuy/surgent/data/processed/CVS_Challenge_SAGES_v1/cvs_act_surgeon_annotations/selected_video_clips.csv
clip JSON root: /mnt/md0/weiqiuy/surgent/data/processed/CVS_Challenge_SAGES_v1/cvs_act_surgeon_annotations/clips
design effect: 1.491 ok


In [3]:
marginals = summarize_marginals(sample_30, pool_138, eligible_168)
marginals


,field,value,sample_30_n,sample_30_pct,pool_138_n,pool_138_pct,pool_168_n,pool_168_pct
0,stratum,"(0,0,1)",3,0.100000,8,0.057971,10,0.059524
1,stratum,"(0,1,0)",7,0.233333,44,0.318841,48,0.285714
2,stratum,"(0,1,1)",4,0.133333,17,0.123188,24,0.142857
3,stratum,"(1,0,0)",3,0.100000,6,0.043478,8,0.047619
4,stratum,"(1,0,1)",2,0.066667,2,0.014493,2,0.011905
5,stratum,"(1,1,0)",6,0.200000,33,0.239130,43,0.255952
6,stratum,"(1,1,1)",5,0.166667,28,0.202899,33,0.196429
7,country_collapsed,0,7,0.233333,32,0.231884,42,0.250000
8,country_collapsed,6,8,0.266667,36,0.260870,43,0.255952
9,country_collapsed,other,15,0.500000,70,0.507246,83,0.494048


In [4]:
per_cell_counts(sample_30)


,table,stratum,country_collapsed,n
0,stratum,"(0,0,1)",,3
1,stratum,"(0,1,0)",,7
2,stratum,"(0,1,1)",,4
3,stratum,"(1,0,0)",,3
4,stratum,"(1,0,1)",,2
5,stratum,"(1,1,0)",,6
6,stratum,"(1,1,1)",,5
7,stratum_x_country,"(0,0,1)",6,1
8,stratum_x_country,"(0,0,1)",other,2
9,stratum_x_country,"(0,1,0)",0,1


In [5]:
selected_video_clips.head()


,clip_id,video_name,criterion,granularity,start,mid,end,coarse_clip_id,stratum,country,country_collapsed,ioc,icg,robotic,device,weight,selected_clip_seed,selected_clip_granularity_pool
0,35ebdf31-e51a-41cb-b455-3afe865d89f1__C3__avg_...,35ebdf31-e51a-41cb-b455-3afe865d89f1,C3,coarse,1650,1800.0,1950,NaN,"(0,0,1)",22,other,0,0,1,4,3.000000,20260617,coarse
1,37419fbb-308e-4362-bcb4-e19b431179b2__C1__avg_...,37419fbb-308e-4362-bcb4-e19b431179b2,C1,coarse,0,150.0,300,NaN,"(1,1,1)",22,other,0,0,0,-1,14.000000,20260617,coarse
2,52fa51d6-7b5e-44bf-a10c-39bf2022068f__C2__avg_...,52fa51d6-7b5e-44bf-a10c-39bf2022068f,C2,coarse,1950,2250.0,2400,NaN,"(0,1,0)",21,other,0,0,0,2,4.000000,20260617,coarse
3,5eb56c91-26a5-40fc-995d-2694c09b8643__C2__avg_...,5eb56c91-26a5-40fc-995d-2694c09b8643,C2,coarse,2100,2250.0,2250,NaN,"(0,1,0)",21,other,0,0,0,2,4.000000,20260617,coarse
4,65f40922-5a16-4929-af41-f9cadd75d5fb__C2__avg_...,65f40922-5a16-4929-af41-f9cadd75d5fb,C2,coarse,900,1200.0,1500,NaN,"(1,1,0)",3,other,0,0,0,0,5.333333,20260617,coarse
